# FULL ADJUSTABLE TRANSIT TOOL

### (fill in necessary links/values and run all)

In [177]:
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
from folium.plugins import HeatMap
import xgboost as xgb
import matplotlib.pyplot as plt
import warnings
import sys
from sklearn.linear_model import LinearRegression
sys.path.append('../..')
from config import CENSUS_KEY
warnings.filterwarnings('ignore')
MODEL_PATH  = '../../models/xgb_model_2.json' 

CENSUS_VARIABLES = [
    'B01003_001E',  # total population
    'B08201_001E',  # total households
    'B08201_002E',  # zero-vehicle households
    'B08201_003E',  # one-vehicle households
    'B08301_001E',  # total commuters
    'B08301_002E',  # drove alone
    'B08301_004E',  # carpooled
    'B08301_010E',  # public transit
    'B23025_001E',  # population 16+ (labor force universe)
    'B23025_004E',  # employed civilians
    'B25003_001E',  # total occupied housing units
    'B25003_003E',  # renter-occupied
    'B19013_001E',  # median household income
]

COUNTIES = [
    #Add county FIPS codes here (last 3 digits of the 5-digit code)
    # '001',  # Adams County
    # '005',  # Arapahoe County
    # # '013',  # Boulder County
    # # '014',  # Broomfield County
    # '059',  # Jefferson County
    # '031',  # Denver County

    '001',  # Alameda
    '075',  # San Francisco
    '081',  # San Mateo
    '085',  # Santa Clara
    '013',  # Contra Costa

]
GRID_SPACING_KM  = 0.5       # distance between candidate grid points
N_STATIONS       = 25        # station budget
MIN_SPACING_KM   = 3.0       # minimum distance between any two selected stations

MIN_JOB_KM = 2.0 #minimum distance between job centers

# BOUND BOX AROUND AREA OF INTEREST (min_lon, min_lat, max_lon, max_lat)
# BOUNDS = (-123.55, 36.85, -121.20, 38.85) 
# for Silicon Valley
BOUNDS = (-122.75, 37.15, -121.55, 38.05) 
# for San Francisco Bay Area
# BOUNDS = (-105.35, 39.55, -104.60, 40.25) 
# for Denver Metro Area


STATE_CODE = '06' # 2-digit state FIPS code (06 for California, 08 for Colorado etc)
STATE_ABBV = 'ca' # LOWERCASE state abbreviation (ca for California, co for Colorado etc)

MAX_LINES = 5 # max number of train lines
# Training features
FEATURE_COLS = [
    'Month',
    'Week',
    'population_density',
    'low_car_rate',
    'transit_commuter_rate',
    'driver_rate',
    'carpool_rate',
    'employment_rate',
    'renter_rate',
    'median_income',
    'is_post_covid',
    'dist_to_nearest_job_center_km'
]

#Census Features
CENSUS_FEATURE_COLS = [
    'population_density',
    'low_car_rate',
    'transit_commuter_rate',
    'driver_rate',
    'carpool_rate',
    'employment_rate',
    'renter_rate',
    'median_income',
]

### Pulling ACS Census Data

In [178]:
vars_str = ','.join(CENSUS_VARIABLES)
county_str = '&in=county:' + ','.join(COUNTIES)
url = (
    f'https://api.census.gov/data/2023/acs/acs5'
    f'?get=NAME,{vars_str}'
    f'&for=tract:*'
    f'&in=state:{STATE_CODE}{county_str}'
    f'&key={CENSUS_KEY}'
)

print('Pulling ACS 2023 data for state tracts...')
response = requests.get(url)
data = response.json()

state_tracts = pd.DataFrame(data[1:], columns=data[0])
state_tracts = state_tracts[state_tracts['county'].isin(COUNTIES)]

for col in CENSUS_VARIABLES:
    state_tracts[col] = pd.to_numeric(state_tracts[col], errors='coerce')
state_tracts.replace(-666666666, np.nan, inplace=True)

state_tracts['GEOID'] = state_tracts['state'] + state_tracts['county'] + state_tracts['tract']
print(f'Tracts loaded: {state_tracts.shape[0]}')
state_tracts.head()

Pulling ACS 2023 data for state tracts...
Tracts loaded: 1447


,NAME,B01003_001E,B08201_001E,B08201_002E,B08201_003E,B08301_001E,B08301_002E,B08301_004E,B08301_010E,B23025_001E,B23025_004E,B25003_001E,B25003_003E,B19013_001E,state,county,tract,GEOID
0,Census Tract 4001; Alameda County; California,3094,1316,85,434,1525,700,126,99,2557,1560,1316,109,250001.0,06,001,400100,06001400100
1,Census Tract 4002; Alameda County; California,2093,861,95,391,1065,328,57,190,1788,1065,861,355,225880.0,06,001,400200,06001400200
2,Census Tract 4003; Alameda County; California,5727,2713,416,1516,3311,1177,86,858,4843,3401,2713,1632,157731.0,06,001,400300,06001400300
3,Census Tract 4004; Alameda County; California,4395,1803,204,890,2375,751,116,426,3410,2459,1803,827,159612.0,06,001,400400,06001400400
4,Census Tract 4005; Alameda County; California,3822,1655,169,711,2129,879,81,297,3294,2137,1655,871,96250.0,06,001,400500,06001400500


### Load TIGER tract boundaries and join demographics

In [179]:
print('Downloading TIGER 2023 tract boundaries (CA)...')
tract_url = 'https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_' + STATE_CODE + '_tract.zip'

state_tract_geo = gpd.read_file(tract_url)
state_tract_geo = state_tract_geo[state_tract_geo['COUNTYFP'].isin(COUNTIES)]
state_tract_geo = state_tract_geo.to_crs('EPSG:4326')
print(f'Tract geometries: {state_tract_geo.shape[0]}')

tracts = state_tract_geo[['GEOID', 'ALAND', 'geometry']].merge(
    state_tracts[['GEOID'] + CENSUS_VARIABLES],
    on='GEOID', how='left'
)
tracts.head()

Tract geometries: 1447


,GEOID,ALAND,geometry,B01003_001E,B08201_001E,B08201_002E,B08201_003E,B08301_001E,B08301_002E,B08301_004E,B08301_010E,B23025_001E,B23025_004E,B25003_001E,B25003_003E,B19013_001E
0,06001442700,1234016,"POLYGON ((-122.01721 37.53932, -122.01719 37.5...",2998,1081,46,297,1381,1014,171,10,2390,1496,1081,246,199154.0
1,06001442800,1278646,"POLYGON ((-122.0023 37.52984, -122.00224 37.52...",3087,944,4,194,1484,938,85,72,2537,1643,944,195,180800.0
2,06001442900,2066523,"POLYGON ((-121.98773 37.52248, -121.98764 37.5...",7942,2460,39,746,3788,2229,223,127,6133,4269,2460,1074,169947.0
3,06001443001,768730,"POLYGON ((-121.9576 37.52392, -121.95746 37.52...",2721,869,37,252,1339,939,45,34,2136,1454,869,119,154178.0
4,06001443002,1580322,"POLYGON ((-121.969 37.52248, -121.96885 37.522...",6915,2001,24,494,3163,2266,141,103,5548,3487,2001,510,141696.0


#### Engineer Model Features

In [180]:
def engineer_features(df):
    out = df.copy()
    out['population_density']    = out['B01003_001E'] / (out['ALAND'] / 1_000_000)
    out['low_car_rate']          = (out['B08201_002E'] + out['B08201_003E']) / out['B08201_001E']
    out['transit_commuter_rate'] = out['B08301_010E'] / out['B08301_001E']
    out['driver_rate']           = out['B08301_002E'] / out['B08301_001E']
    out['carpool_rate']          = out['B08301_004E'] / out['B08301_001E']
    out['employment_rate']       = out['B23025_004E'] / out['B23025_001E']
    out['renter_rate']           = out['B25003_003E'] / out['B25003_001E']
    out['median_income']         = out['B19013_001E']
    return out

tracts = engineer_features(tracts)
tracts[CENSUS_FEATURE_COLS].describe()

,population_density,low_car_rate,transit_commuter_rate,driver_rate,carpool_rate,employment_rate,renter_rate,median_income
count,1442.000000,1437.000000,1437.000000,1437.000000,1437.000000,1439.000000,1437.000000,1429.000000
mean,4981.007614,0.430791,0.088056,0.612530,0.082120,0.636856,0.445567,148652.389783
std,5915.393129,0.227279,0.089465,0.194835,0.055741,0.089734,0.252301,56648.099504
min,0.000000,0.037736,0.000000,0.000000,0.000000,0.000000,0.023336,13569.000000
25%,2011.692200,0.258877,0.023438,0.506494,0.041374,0.589714,0.220853,105686.000000
50%,3451.378779,0.377907,0.055484,0.652850,0.071184,0.634667,0.421309,143665.000000
75%,6026.796670,0.578947,0.128205,0.757382,0.113032,0.689602,0.651795,188098.000000
max,76523.867809,1.000000,0.796020,0.960837,0.378253,1.000000,1.000000,250001.000000


In [181]:
def dist_to_point(lat, lon, clat, clon):
    R = 6371
    dlat, dlon = np.radians(lat - clat), np.radians(lon - clon)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat))*np.cos(np.radians(clat))*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def dist_to_nearest_job_center(lat, lon, centers):
    dists = [dist_to_point(lat, lon, c[0], c[1]) for c in centers]
    return min(dists)


### Load Job-center Data (LODES)

In [182]:
lodes_url = 'https://lehd.ces.census.gov/data/lodes/LODES8/' + STATE_ABBV + '/wac/' + STATE_ABBV + '_wac_S000_JT00_2021.csv.gz'

lodes = pd.read_csv(lodes_url, compression='gzip', dtype={'w_geocode': str})

# w_geocode is an 11-digit census block — first 11 chars = tract GEOID
lodes['GEOID'] = lodes['w_geocode'].str[:11]

# Aggregate jobs to tract level
jobs_by_tract = lodes.groupby('GEOID')['C000'].sum().reset_index()
jobs_by_tract.rename(columns={'C000': 'total_jobs'}, inplace=True)

print(jobs_by_tract.sort_values('total_jobs', ascending=False).head(10))

            GEOID  total_jobs
1977  06037207400      123890
6529  06073008511      106997
7061  06075011700       88928
2528  06037310400       85246
4600  06059075516       79766
4469  06059062610       78095
7980  06085505006       67760
7269  06075061501       61137
3848  06037980013       56240
2345  06037267901       54447


In [183]:
tracts = tracts.merge(jobs_by_tract, on='GEOID', how='left')
tracts['job_density'] = tracts['total_jobs'] / (tracts['ALAND'] / 1_000_000)

#### Filter Job Centers

In [184]:
emp_threshold = np.percentile(tracts['job_density'].dropna(), 95)
job_dense_tracts = tracts[tracts['job_density'] >= emp_threshold]
print(f'Tracts above 95th percentile of job density: {len(job_dense_tracts)}')

filtered_centers = []

for _, row in job_dense_tracts.sort_values('job_density', ascending=False).iterrows():
    lat = row.geometry.centroid.y
    lon = row.geometry.centroid.x
    
    if not filtered_centers:
        filtered_centers.append((lat, lon))
        continue
    
    # compute distance to all already accepted centers
    dists = [
        dist_to_point(lat, lon, c[0], c[1])
        for c in filtered_centers
    ]
    
    if min(dists) >= MIN_JOB_KM:
        filtered_centers.append((lat, lon))

job_centers = np.array(filtered_centers)

print(f'Filtered job centers: {len(job_centers)}')
print(job_centers[:10])


Tracts above 95th percentile of job density: 73
Filtered job centers: 20
[[  37.79049752 -122.40347484]
 [  37.80205488 -122.26507303]
 [  37.77133088 -122.42901331]
 [  37.76425678 -122.45974279]
 [  37.87523964 -122.26958779]
 [  37.82331259 -122.2536628 ]
 [  37.44800754 -122.16221258]
 [  37.34990879 -121.90216989]
 [  37.33036342 -122.00488747]
 [  37.33061558 -121.89555657]]


### Generate Candidate Grid

In [185]:
def generate_grid(bounds, spacing_km):
    min_lon, min_lat, max_lon, max_lat = bounds
    deg_lat = spacing_km / 110.574
    deg_lon = spacing_km / (111.320 * np.cos(np.radians((min_lat + max_lat) / 2)))
    lats = np.arange(min_lat, max_lat, deg_lat)
    lons = np.arange(min_lon, max_lon, deg_lon)
    grid_lons, grid_lats = np.meshgrid(lons, lats)
    pts = pd.DataFrame({'lon': grid_lons.ravel(), 'lat': grid_lats.ravel()})
    return gpd.GeoDataFrame(
        pts,
        geometry=[Point(x, y) for x, y in zip(pts.lon, pts.lat)],
        crs='EPSG:4326'
    )

grid = generate_grid(BOUNDS, GRID_SPACING_KM)
print(f'Total grid points: {len(grid)}')

# Spatial join: each grid point inherits features of the tract it falls inside
grid_joined = gpd.sjoin(
    grid,
    tracts[['GEOID', 'ALAND', 'geometry'] + CENSUS_FEATURE_COLS],
    how='left',
    predicate='within'
)
# Drop points outside county boundary
grid_joined = grid_joined.dropna(subset=['GEOID']).reset_index(drop=True)
grid_joined['is_post_covid'] = 1
grid_joined['Month']                   = 6
grid_joined['Week']                    = 24
print(f'Candidates inside bounds: {len(grid_joined)}')
grid_joined['dist_to_nearest_job_center_km'] = grid_joined.apply(
    lambda r: dist_to_nearest_job_center(r.lat, r.lon, job_centers), axis=1
)

# Fill sparse NaNs with tract medians
for col in FEATURE_COLS:
    grid_joined[col] = grid_joined[col].fillna(grid_joined[col].median())

grid_joined[['lon','lat'] + FEATURE_COLS].head()

Total grid points: 42400
Candidates inside bounds: 32505


,lon,lat,Month,Week,population_density,low_car_rate,transit_commuter_rate,driver_rate,carpool_rate,employment_rate,renter_rate,median_income,is_post_covid,dist_to_nearest_job_center_km
0,-122.443870,37.15,6,24,121.102984,0.228835,0.032997,0.681026,0.075688,0.610733,0.242583,175159.0,1,41.457692
1,-122.438200,37.15,6,24,121.102984,0.228835,0.032997,0.681026,0.075688,0.610733,0.242583,175159.0,1,41.158303
2,-122.432531,37.15,6,24,121.102984,0.228835,0.032997,0.681026,0.075688,0.610733,0.242583,175159.0,1,40.862874
3,-122.426862,37.15,6,24,121.102984,0.228835,0.032997,0.681026,0.075688,0.610733,0.242583,175159.0,1,40.571492
4,-122.421193,37.15,6,24,121.102984,0.228835,0.032997,0.681026,0.075688,0.610733,0.242583,175159.0,1,40.284244


## Predict Using Model

In [186]:
model = xgb.XGBRegressor()
model.load_model(MODEL_PATH)

X = grid_joined[FEATURE_COLS].values
grid_joined['predicted_ridership'] = np.expm1(model.predict(X))

print(f"Score range: {grid_joined['predicted_ridership'].min():.0f} – {grid_joined['predicted_ridership'].max():.0f}")
grid_joined[['lon','lat','predicted_ridership']].sort_values('predicted_ridership', ascending=False).head(10)


Score range: 3430 – 77886


,lon,lat,predicted_ridership
23186,-122.409855,37.778538,77886.406250
23369,-122.404186,37.783060,77886.406250
23734,-122.398517,37.792104,70785.609375
23551,-122.404186,37.787582,70785.609375
23733,-122.404186,37.792104,70342.000000
23918,-122.392848,37.796626,67915.789062
24286,-122.398517,37.805670,67915.789062
23917,-122.398517,37.796626,67915.789062
24101,-122.398517,37.801148,67915.789062
24102,-122.392848,37.801148,67915.789062


## Greedy Station Selection

In [187]:
def haversine(lat1, lon1, lat2_arr, lon2_arr):
    R = 6371.0
    dlat = np.radians(lat2_arr - lat1)
    dlon = np.radians(lon2_arr - lon1)
    a = (np.sin(dlat/2)**2
         + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2_arr)) * np.sin(dlon/2)**2)
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def greedy_select(candidates, n, min_spacing_km):
    ranked = candidates.sort_values('predicted_ridership', ascending=False)
    selected = []
    sel_lats, sel_lons = [], []

    for idx, row in ranked.iterrows():
        if len(selected) >= n:
            break
        if sel_lats:
            dists = haversine(row['lat'], row['lon'],
                              np.array(sel_lats), np.array(sel_lons))
            if dists.min() < min_spacing_km:
                continue
        selected.append(idx)
        sel_lats.append(row['lat'])
        sel_lons.append(row['lon'])

    return selected

selected_idx = greedy_select(grid_joined, N_STATIONS, MIN_SPACING_KM)
selected = grid_joined.loc[selected_idx].reset_index(drop=True)

print(f'Selected {len(selected)} stations')
print(f'Total predicted ridership: {selected["predicted_ridership"].sum():.0f}')
selected[['lat','lon','predicted_ridership'] + CENSUS_FEATURE_COLS].round(4)

Selected 25 stations
Total predicted ridership: 813515


,lat,lon,predicted_ridership,population_density,low_car_rate,transit_commuter_rate,driver_rate,carpool_rate,employment_rate,renter_rate,median_income
0,37.7785,-122.4099,77886.406250,11898.6184,0.9276,0.1528,0.1598,0.0551,0.6654,0.9113,83468.0
1,37.8057,-122.3985,67915.789062,4699.0855,0.9557,0.1797,0.1595,0.0483,0.5876,0.7906,171667.0
2,37.8690,-122.2568,64708.753906,22475.8977,0.9568,0.1987,0.1329,0.0078,0.3716,0.9907,30833.0
3,37.4258,-122.1831,63916.203125,1316.7822,0.9279,0.0343,0.1784,0.0448,0.4227,1.0000,44861.0
4,37.8057,-122.2738,40818.328125,12891.9508,0.9640,0.4269,0.2134,0.0551,0.4000,0.9511,18552.0
5,37.7695,-122.4495,40318.574219,14289.6202,0.9079,0.2308,0.2459,0.0599,0.8833,0.7996,218394.0
6,37.7966,-122.4382,32225.755859,10194.8289,0.7786,0.1960,0.1716,0.0281,0.8206,0.7508,250001.0
7,37.7785,-122.4836,31438.988281,13001.4696,0.7131,0.2389,0.3239,0.0357,0.6368,0.5817,116750.0
8,37.7243,-122.4212,31082.748047,105.3321,0.9675,0.0000,0.5556,0.0000,0.0581,1.0000,13569.0
9,37.8011,-122.3362,30795.855469,2.3531,1.0000,0.0000,0.5556,0.0000,1.0000,1.0000,175159.0


## Route Generation (TODO)

In [188]:
from scipy.optimize import linprog
import itertools
PERP_KM = 1.5

In [189]:
def straightness_score(lats, lons):
    if len(lats) < 2:
        return 0
    if len(np.unique(lons)) == 1:  # perfectly vertical line
        return 1.0
    if len(np.unique(lats)) == 1:  # perfectly horizontal line
        return 1.0
    # Try both orientations and take the best
    r2_a = LinearRegression().fit(lons.reshape(-1,1), lats).score(lons.reshape(-1,1), lats)
    r2_b = LinearRegression().fit(lats.reshape(-1,1), lons).score(lats.reshape(-1,1), lons)
    return max(r2_a, r2_b)

In [ ]:
def generate_all_candidate_lines(selected, min_stations=3, perp_threshold_km=PERP_KM):
    """
    Generate every possible straight-ish line through subsets of stations.
    Returns list of (station_indices, score) tuples.
    """
    candidates = []
    n = len(selected)
    coords = selected[['lat', 'lon']].values
    ridership = selected['predicted_ridership'].values
    
    # For each pair of stations, define a line direction
    # then find all stations close to that line
    for i, j in itertools.combinations(range(n), 2):
        # Direction vector between stations i and j
        direction = coords[j] - coords[i]
        norm = np.linalg.norm(direction)
        if norm < 1e-9:
            continue
        direction = direction / norm
        origin = coords[i]
        lat_rad = np.radians(origin[0])
        
        # Find all stations within perp_threshold of this line
        on_line = []
        for k in range(n):
            v = coords[k] - origin
            proj = np.dot(v, direction) * direction
            perp = v - proj
            perp_km = np.sqrt(
                (perp[0] * 110.574)**2 +
                (perp[1] * 111.320 * np.cos(lat_rad))**2
            )
            if perp_km < perp_threshold_km:
                on_line.append(k)
        
        if len(on_line) < min_stations:
            continue
        
        # Score this candidate line
        subset_lats = coords[on_line, 0]
        subset_lons = coords[on_line, 1]
        r2 = straightness_score(subset_lats, subset_lons)
        total_ridership = ridership[on_line].sum()
        score = total_ridership * (r2 ** 2)  # square R² to penalize jaggedness harder
        
        candidates.append((on_line, score))
    
    # Deduplicate — same set of stations from different anchor pairs
    seen = set()
    unique = []
    for station_indices, score in candidates:
        key = frozenset(station_indices)
        if key not in seen:
            seen.add(key)
            unique.append((station_indices, score))
    
    return sorted(unique, key=lambda x: x[1], reverse=True)

def jaccard_similarity(set1, set2):
    intersection = len(set(set1) & set(set2))
    union = len(set(set1) | set(set2))
    return intersection / union

def select_lines_greedy(candidates, selected, max_lines=MAX_LINES,
                        interchange_percentile=60, max_similarity=0.7):
    """
    Greedily pick the best non-redundant lines.
    A line is redundant if >max_similarity% of its stations are already covered.
    High ridership stations can appear on multiple lines (interchanges).
    """
    interchange_threshold = np.percentile(
        selected['predicted_ridership'], interchange_percentile
    )
    
    covered = set()
    line_orders = {}
    selected_line_sets = []  # track station sets of chosen lines
    line_id = 0
    
    for station_indices, score in candidates:
        if line_id >= max_lines:
            break
        
        too_similar = any(
            jaccard_similarity(station_indices, existing) > max_similarity
            for existing in selected_line_sets
        )
        if too_similar:
            continue
        
        # Skip if too many non-interchange stations already covered
        non_interchange_covered = [
            i for i in station_indices
            if i in covered and 
            selected.iloc[i]['predicted_ridership'] < interchange_threshold
        ]
        if len(non_interchange_covered) / len(station_indices) > 0.5:
            continue
        
         # Order stations along the line
        coords = selected[['lat', 'lon']].values
        subset_coords = coords[station_indices]
        centered = subset_coords - subset_coords.mean(axis=0)
        _, _, Vt = np.linalg.svd(centered)
        direction = Vt[0]
        projections = centered @ direction
        order = np.argsort(projections)
        ordered_indices = [station_indices[i] for i in order]
        df_indices = selected.iloc[ordered_indices].index.tolist()
        
        line_orders[line_id] = df_indices
        selected_line_sets.append(set(station_indices))
        
        # Mark non-interchange stations as covered
        for i in station_indices:
            if selected.iloc[i]['predicted_ridership'] < interchange_threshold:
                covered.add(i)
        
        r2 = straightness_score(
            selected.iloc[ordered_indices]['lat'].values,
            selected.iloc[ordered_indices]['lon'].values
        )
        ridership = selected.iloc[ordered_indices]['predicted_ridership'].sum()
        print(f'Line {line_id+1}: {len(ordered_indices)} stations | '
              f'R²={r2:.3f} | ridership={ridership:.0f} | score={score:.0f}')
        
        line_id += 1
    
    return line_orders

In [191]:
print('Generating candidate lines...')
candidates = generate_all_candidate_lines(
    selected, 
    min_stations=3, 
    perp_threshold_km=PERP_KM
)
print(f'Unique candidate lines: {len(candidates)}')

print('\nSelecting best lines...')
line_orders = select_lines_greedy(
    candidates, selected, MAX_LINES
)

print(f'\nTotal lines selected: {len(line_orders)}')

for line_id, station_idx in line_orders.items():
    print(f'\nLine {line_id + 1}:')
    print(selected.loc[station_idx][['lat', 'lon', 'predicted_ridership']].to_string())

Generating candidate lines...
Unique candidate lines: 137

Selecting best lines...
Line 1: 5 stations | R²=0.973 | ridership=272622 | score=258159
Line 2: 6 stations | R²=0.964 | ridership=258357 | score=240154
Line 3: 6 stations | R²=0.976 | ridership=251051 | score=239117
Line 4: 6 stations | R²=0.969 | ridership=250880 | score=235709
Line 5: 5 stations | R²=0.987 | ridership=229254 | score=223279

Total lines selected: 5

Line 1:
          lat         lon  predicted_ridership
16  37.751407 -122.483553         21792.134766
5   37.769495 -122.449539         40318.574219
0   37.778538 -122.409855         77886.406250
1   37.805670 -122.398517         67915.789062
2   37.868976 -122.256790         64708.753906

Line 2:
          lat         lon  predicted_ridership
15  37.697145 -122.483553         22611.507812
11  37.719754 -122.460877         28485.048828
5   37.769495 -122.449539         40318.574219
0   37.778538 -122.409855         77886.406250
1   37.805670 -122.398517         679

## MAPPING

In [ ]:
LINE_COLORS = [
    '#e41a1c','#377eb8','#4daf4a','#ff7f00',
    '#984ea3','#a65628','#f781bf','#999999',
    '#00ced1','#ffd700'
]

cmap = plt.cm.YlOrRd
vmin = selected['predicted_ridership'].min()
vmax = selected['predicted_ridership'].max()

def score_to_hex(val):
    norm = (val - vmin) / (vmax - vmin + 1e-9)
    r, g, b, _ = cmap(norm)
    return '#{:02x}{:02x}{:02x}'.format(int(r*255), int(g*255), int(b*255))

m = folium.Map(location=[37.33, -121.89], zoom_start=11, tiles='CartoDB positron')

# Background heatmap
heat_data = grid_joined[['lat','lon','predicted_ridership']].values.tolist()
HeatMap(heat_data, radius=14, blur=18, min_opacity=0.3,
        gradient={'0.3':'blue','0.55':'lime','0.75':'orange','1.0':'red'}).add_to(m)

# Draw lines first so markers render on top
for line_id, station_idx in line_orders.items():
    color = LINE_COLORS[line_id % len(LINE_COLORS)]
    line_stations = selected.loc[station_idx]
    coords = [[row['lat'], row['lon']] for _, row in line_stations.iterrows()]
    folium.PolyLine(
        coords,
        color=color,
        weight=5,
        opacity=0.8,
        tooltip=f'Line {line_id + 1}'
    ).add_to(m)

# Station markers — color by line, size by ridership
for line_id, station_idx in line_orders.items():
    color = LINE_COLORS[line_id % len(LINE_COLORS)]
    line_stations = selected.loc[station_idx]
    for i, (idx, row) in enumerate(line_stations.iterrows()):
        is_terminal = (i == 0 or i == len(line_stations) - 1)
        is_interchange = selected.loc[idx, 'is_interchange'] if 'is_interchange' in selected.columns else False
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=12 if is_interchange else (10 if is_terminal else 7),
            color='white', weight=2,
            fill=True, fill_color=color, fill_opacity=0.95,
            tooltip=(
                f"<b>Line {line_id+1} — Station {i+1}</b><br>"
                f"{'🔀 INTERCHANGE — ' if is_interchange else ''}"
                f"{'⬤ TERMINAL — ' if is_terminal else ''}"
                f"Predicted ridership: <b>{row['predicted_ridership']:.0f}</b><br>"
                f"Pop density: {row['population_density']:.0f}/km²<br>"
                f"Transit rate: {row['transit_commuter_rate']:.1%}<br>"
                f"Low-car rate: {row['low_car_rate']:.1%}<br>"
                f"Employment rate: {row['employment_rate']:.1%}<br>"
                f"Renter rate: {row['renter_rate']:.1%}<br>"
                f"Median income: ${row['median_income']:,.0f}<br>"
                f"Job proximity: {row['dist_to_nearest_job_center_km']:.1f} km"
            )
        ).add_to(m)

# Legend
legend_items = ''.join([
    f"<span style='color:{LINE_COLORS[i % len(LINE_COLORS)]}'>━━</span> Line {i+1}&nbsp;&nbsp;"
    for i in range(len(line_orders))
])
legend_html = f"""
<div style='position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:12px 16px;border-radius:8px;
     box-shadow:2px 2px 8px rgba(0,0,0,.3);font-family:sans-serif;font-size:13px;max-width:400px'>
  <b>Transit Lines</b><br>{legend_items}<br><br>
  <span style='font-size:11px'>
  ⬤ Terminal &nbsp;|&nbsp; 🔀 Interchange &nbsp;|&nbsp; Hover for details
  </span>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m.save('transit_stations.html')
print('Saved → transit_stations.html')

Saved → transit_stations.html
